In [25]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import logging
import time
import datetime

spark = (
    SparkSession.builder
    .appName("Lesson26-Postgres-Iceberg")
    .master("spark://spark-master:7077")

    .config("spark.executor.cores", "1")
    .config("spark.executor.memory", "2g")
    .config("spark.executor.memoryOverhead", "512m")
    .config("spark.driver.memory", "1g")

    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "matrix")
    .config("spark.hadoop.fs.s3a.secret.key", "matrix123")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")

    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.local.type", "hadoop")
    .config("spark.sql.catalog.local.warehouse", "s3a://matrix/warehouse")
    .getOrCreate()
)

print("executor.cores:", spark.conf.get("spark.executor.cores"))
print("executor.memory:", spark.conf.get("spark.executor.memory"))
print("executor.memoryOverhead:", spark.conf.get("spark.executor.memoryOverhead"))

executor.cores: 1
executor.memory: 2g
executor.memoryOverhead: 512m


## 1. Logging (Slayd 5)


In [26]:
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("orders_pipeline")

batch_id = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
logger.info(f"Pipeline  batch_id={batch_id}")

2026-09-26 07:18:08,657 [INFO] Pipeline  batch_id=20260926_071808


## 2. Read — Postgres


In [ ]:
jdbc_url = "jdbc:postgresql://postgres:5432/lesson23"
pg_props = {"user": "matrix", "password": "matrix123", "driver": "org.postgresql.Driver"}

t0 = time.time()
raw_orders = spark.read.jdbc(
    url=jdbc_url,
    table="orders",
    column="order_id",
    lowerBound=1,
    upperBound=2_000_000,
    numPartitions=4,
    properties=pg_props,
)
row_count = raw_orders.count()
logger.info(f"Postgres read: {row_count:,} row, {raw_orders.rdd.getNumPartitions()} partition, {time.time()-t0:.1f} sec.")
raw_orders.printSchema()
raw_orders.show(5)

2026-09-26 07:18:11,866 [INFO] Postgres read: 2,000,000 sətir, 4 partition, 1.4 sec.


root
 |-- order_id: long (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- amount: decimal(10,2) (nullable = true)
 |-- order_date: date (nullable = true)
 |-- region: string (nullable = true)

+--------+-----------+------+----------+--------+
|order_id|customer_id|amount|order_date|  region|
+--------+-----------+------+----------+--------+
|       1|          2|434.92|2026-01-02|   Ganja|
|       2|          3| 48.03|2026-01-03|Sumqayit|
|       3|          4|335.78|2026-01-04|Sumqayit|
|       4|          5|988.05|2026-01-05|Sumqayit|
|       5|          6| 71.31|2026-01-06|    Baki|
+--------+-----------+------+----------+--------+
only showing top 5 rows



## 3. Transform 



In [28]:
transformed = (
    raw_orders
    .withColumn("amount_category", F.when(F.col("amount") < 100, "low")
                                     .when(F.col("amount") < 500, "medium")
                                     .otherwise("high"))
    .withColumn("processed_at", F.current_timestamp())
    .withColumn("batch_id", F.lit(batch_id))
    .select("order_id", "customer_id", "amount", "amount_category",
            "order_date", "processed_at", "batch_id")
)
transformed.printSchema()
transformed.show(5)

root
 |-- order_id: long (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- amount: decimal(10,2) (nullable = true)
 |-- amount_category: string (nullable = false)
 |-- order_date: date (nullable = true)
 |-- processed_at: timestamp (nullable = false)
 |-- batch_id: string (nullable = false)

+--------+-----------+------+---------------+----------+--------------------+---------------+
|order_id|customer_id|amount|amount_category|order_date|        processed_at|       batch_id|
+--------+-----------+------+---------------+----------+--------------------+---------------+
|       1|          2|434.92|         medium|2026-01-02|2026-09-26 07:18:...|20260926_071808|
|       2|          3| 48.03|            low|2026-01-03|2026-09-26 07:18:...|20260926_071808|
|       3|          4|335.78|         medium|2026-01-04|2026-09-26 07:18:...|20260926_071808|
|       4|          5|988.05|           high|2026-01-05|2026-09-26 07:18:...|20260926_071808|
|       5|          6| 71.31|   

## 4. Cache 


In [29]:
t0 = time.time()
transformed.cache()
transformed.count()  
logger.info(f"DataFrame cache : {time.time()-t0:.1f} sec.")

2026-09-26 07:18:31,554 [INFO] DataFrame cache : 3.5 sec.


## 5. Iceberg 


In [30]:
# spark.sql("CREATE DATABASE IF NOT EXISTS local.db")

spark.sql("""
CREATE TABLE IF NOT EXISTS local.db.orders_analytics (
    order_id STRING, customer_id STRING, amount DOUBLE,
    amount_category STRING, order_date DATE,
    processed_at TIMESTAMP, batch_id STRING
)
USING iceberg
PARTITIONED BY (days(order_date))
TBLPROPERTIES ('write.target-file-size-bytes'='134217728')
""")


DataFrame[]

In [24]:
spark.sql("drop table local.db.orders_analytics")

DataFrame[]

## 6. Error Handling + Retry + Idempotent Processing 




In [31]:
def write_to_iceberg_idempotent(df, max_retries=3):
    df.createOrReplaceTempView("staging_orders")
    for attempt in range(1, max_retries + 1):
        try:
            spark.sql("""
                MERGE INTO local.db.orders_analytics t
                USING staging_orders s
                ON t.order_id = s.order_id
                WHEN MATCHED THEN UPDATE SET
                    t.amount = s.amount, t.amount_category = s.amount_category,
                    t.processed_at = s.processed_at, t.batch_id = s.batch_id
                WHEN NOT MATCHED THEN INSERT (order_id, customer_id, amount, amount_category, order_date, processed_at, batch_id)
                    VALUES (s.order_id, s.customer_id, s.amount, s.amount_category, s.order_date, s.processed_at, s.batch_id)
            """)
            logger.info(f"Iceberg MERGE succes (retry {attempt})")
            return True
        except Exception as e:
            logger.warning(f"Iceberg error , retry {attempt}/{max_retries}: {e}")
            if attempt == max_retries:
                logger.error(f"Iceberg failed, batch_id={batch_id}")
                raise
            time.sleep(2 ** attempt)

write_to_iceberg_idempotent(transformed)
spark.table("local.db.orders_analytics").count()

2026-09-26 07:18:49,473 [INFO] Iceberg MERGE succes (retry 1)


2000000

### İdempotent



In [32]:
before = spark.table("local.db.orders_analytics").count()
write_to_iceberg_idempotent(transformed)
after = spark.table("local.db.orders_analytics").count()

print(f" first: {before:,}")
print(f"second:  {after:,}  ")
assert before == after, "false"
print("✓ true.")

2026-09-26 07:19:59,594 [INFO] Iceberg MERGE succes (retry 1)


 firs: 2,000,000
second:  2,000,000  
✓ true.


## 7. Metadata 



In [33]:
import datetime


print("Snapshot sayı:", spark.sql("SELECT COUNT(*) as cnt FROM local.db.orders_analytics.snapshots").collect()[0].cnt)
print("Data fayl sayı:", spark.sql("SELECT COUNT(*) as cnt FROM local.db.orders_analytics.files").collect()[0].cnt)

Snapshot sayı: 2
Data fayl sayı: 365


In [34]:
now = datetime.datetime.now().isoformat(sep=" ", timespec="seconds")

spark.sql(f"""
CALL local.system.expire_snapshots(
    table => 'db.orders_analytics',
    older_than => TIMESTAMP '{now}',
    retain_last => 1
)
""").show(truncate=False)

+------------------------+-----------------------------------+-----------------------------------+----------------------------+----------------------------+------------------------------+
|deleted_data_files_count|deleted_position_delete_files_count|deleted_equality_delete_files_count|deleted_manifest_files_count|deleted_manifest_lists_count|deleted_statistics_files_count|
+------------------------+-----------------------------------+-----------------------------------+----------------------------+----------------------------+------------------------------+
|365                     |0                                  |0                                  |1                           |1                           |0                             |
+------------------------+-----------------------------------+-----------------------------------+----------------------------+----------------------------+------------------------------+



In [35]:
# 2) remove_orphan_files 
spark.sql("""
CALL local.system.remove_orphan_files(
    table => 'db.orders_analytics'
)
""").show(truncate=False)

+--------------------+
|orphan_file_location|
+--------------------+
+--------------------+



In [36]:
# 3) rewrite_data_files —
spark.sql("""
CALL local.system.rewrite_data_files(
    table => 'db.orders_analytics',
    options => map('rewrite-all', 'true')
)
""").show(truncate=False)

+--------------------------+----------------------+---------------------+-----------------------+--------------------------+
|rewritten_data_files_count|added_data_files_count|rewritten_bytes_count|failed_data_files_count|removed_delete_files_count|
+--------------------------+----------------------+---------------------+-----------------------+--------------------------+
|365                       |365                   |19792259             |0                      |0                         |
+--------------------------+----------------------+---------------------+-----------------------+--------------------------+



In [37]:
# 4) rewrite_manifests 
spark.sql("""
CALL local.system.rewrite_manifests(
    table => 'db.orders_analytics'
)
""").show(truncate=False)

+-------------------------+---------------------+
|rewritten_manifests_count|added_manifests_count|
+-------------------------+---------------------+
|2                        |1                    |
+-------------------------+---------------------+



In [38]:

print("Snapshot sayı:", spark.sql("SELECT COUNT(*) as cnt FROM local.db.orders_analytics.snapshots").collect()[0].cnt)
print("Data fayl sayı:", spark.sql("SELECT COUNT(*) as cnt FROM local.db.orders_analytics.files").collect()[0].cnt)
print("Cəmi sətir sayı (dəyişməməlidir):", spark.table("local.db.orders_analytics").count())

Snapshot sayı: 3
Data fayl sayı: 365
Cəmi sətir sayı (dəyişməməlidir): 2000000


In [11]:
transformed.unpersist()


DataFrame[order_id: bigint, customer_id: int, amount: decimal(10,2), amount_category: string, order_date: date, processed_at: timestamp, batch_id: string]

In [ ]:
# logger.info(f"Pipeline done. batch_id={batch_id}")
# spark.stop()